# Individual 5-Category Benchmark — Final Winning Architecture (Zero Pooling)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Trains **5 completely separate models** (1 per MVTec LOCO category, strictly single-category `train/good` → single-category `validation/good` calibration → single-category `test/good` vs. `test/logical_anomalies`, **zero pooling**) on the single overall final winning architecture from `05_colab_final_integration.ipynb`.

**Direct Comparison Against Prior Published Individual-Category Baselines (Assignment 2 & 3):**
- A2 **PUAD**: `91.58%` (BB `91.47%`, JB `99.88%`, PP `96.82%`, SB `74.49%`, SC `95.22%`)
- A3 **SALAD**: `91.52%` | **LogiCo**: `87.86%` | **CSAD**: `86.54%` | **GCAD**: `86.52%` | **ComAD**: `76.03%`

In [ ]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (INDIVIDUAL 5-CAT BENCHMARK) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    if os.path.exists('/content/data/breakfast_box'):
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
    else:
        os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
        !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
else:
    os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'

if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

In [ ]:
#@title 3a. Train & Evaluate E21 Individual — Breakfast Box
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --category breakfast_box --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 3b. Train & Evaluate E21 Individual — Juice Bottle
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --category juice_bottle --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 3c. Train & Evaluate E21 Individual — Pushpins
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --category pushpins --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 3d. Train & Evaluate E21 Individual — Screw Bag
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --category screw_bag --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 3e. Train & Evaluate E21 Individual — Splicing Connectors
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --category splicing_connectors --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 4. Compute Individual Mean, Pooled 1,136-Image Joint AUROC & A2-Style Table
import json, pathlib, yaml
import pandas as pd
from sklearn.metrics import roc_auc_score

WINNER_CONFIG = 'configs/E21_fpn_crisscross_transformer.yaml'
CATEGORIES = ['breakfast_box', 'juice_bottle', 'pushpins', 'screw_bag', 'splicing_connectors']
exp_id = yaml.safe_load(open(WINNER_CONFIG))['exp_id']
scores = {}
score_dfs = []

for cat in CATEGORIES:
    local_dir = pathlib.Path('experiments') / f'{exp_id}_ind_{cat}'
    drive_dir = pathlib.Path(DRIVE_RUNS) / f'{exp_id}_ind_{cat}'
    run_dir = local_dir if (local_dir / 'test_results.json').exists() else drive_dir
    d = json.loads((run_dir / 'test_results.json').read_text())
    scores[cat] = d['metrics']['per_category_auroc'][cat]
    score_dfs.append(pd.read_csv(run_dir / 'test_scores.csv'))

# Concatenate all 5 individual models' test predictions (575 good + 561 logical = 1,136 images)
pooled_df = pd.concat(score_dfs, ignore_index=True)
assert len(pooled_df) == 1136, f'Expected 1136 logical-only test images, got {len(pooled_df)}'
assert int((pooled_df['label'] == 0).sum()) == 575
assert int((pooled_df['label'] == 1).sum()) == 561
assert 'structural_anomalies' not in set(pooled_df['anomaly_type'].unique())

mean_ind = sum(scores.values()) / len(scores)
pooled_ind_raw = float(roc_auc_score(pooled_df['label'], pooled_df['score']))
pooled_ind_ea = float(roc_auc_score(pooled_df['label'], pooled_df['efficientad_score']))
pooled_ind_maha = float(roc_auc_score(pooled_df['label'], pooled_df['mahalanobis']))

out_summary_dir = pathlib.Path(DRIVE_RUNS) / 'final_summary'
out_summary_dir.mkdir(parents=True, exist_ok=True)
pooled_csv_path = out_summary_dir / f'{exp_id}_individual_pooled_test_scores.csv'
pooled_df.to_csv(pooled_csv_path, index=False)

summary_payload = {
    'exp_id': exp_id,
    'config': WINNER_CONFIG,
    'n_test_total': len(pooled_df),
    'n_test_good': int((pooled_df['label'] == 0).sum()),
    'n_test_logical': int((pooled_df['label'] == 1).sum()),
    'per_category_auroc': scores,
    'individual_mean_auroc': mean_ind,
    'individual_pooled_raw_auroc': pooled_ind_raw,
    'individual_pooled_efficientad_auroc': pooled_ind_ea,
    'individual_pooled_mahalanobis_auroc': pooled_ind_maha,
    'delta_mean_vs_a2_puad_ind': mean_ind - 0.9158,
    'delta_mean_vs_a3_salad_ind': mean_ind - 0.9152,
    'delta_pooled_vs_a2_puad_ind': pooled_ind_raw - 0.8630,
    'delta_pooled_vs_e21_combined': pooled_ind_raw - 0.8736,
}
summary_json_path = out_summary_dir / f'{exp_id}_individual_summary.json'
summary_json_path.write_text(json.dumps(summary_payload, indent=2))

print('\n' + '=' * 74)
print(f'INDIVIDUAL 5-CATEGORY BENCHMARK (ZERO TRAINING POOLING): {exp_id}')
print('=' * 74)
for cat, v in scores.items():
    print(f'  {cat:34s} : {v * 100:6.2f}%')
print('-' * 74)
print(f'  {"Individual Mean AUROC":34s} : {mean_ind * 100:6.2f}%')
print(f'  {"Individual Pooled Raw AUROC":34s} : {pooled_ind_raw * 100:6.2f}%  (1,136 imgs: 575 good + 561 logical)')
print(f'    - Pooled EfficientAD only        : {pooled_ind_ea * 100:6.2f}%')
print(f'    - Pooled Mahalanobis only        : {pooled_ind_maha * 100:6.2f}%')
print('-' * 74)
print(f'  {"Delta Mean vs A2 PUAD Ind (91.58%)":34s} : {(mean_ind - 0.9158) * 100:+6.2f} pp')
print(f'  {"Delta Mean vs A3 SALAD Ind (91.52%)":34s} : {(mean_ind - 0.9152) * 100:+6.2f} pp')
print(f'  {"Delta Pooled vs A2 PUAD Ind (86.30%)":34s} : {(pooled_ind_raw - 0.8630) * 100:+6.2f} pp')
print(f'  {"Delta Pooled vs E21 Comb (87.36%)":34s} : {(pooled_ind_raw - 0.8736) * 100:+6.2f} pp')
print('=' * 74)

e21_comb = {
    'breakfast_box': 0.9193,
    'juice_bottle': 0.9770,
    'pushpins': 0.9922,
    'screw_bag': 0.7028,
    'splicing_connectors': 0.9526,
    'mean': 0.9088,
    'pooled': 0.8736,
}
print('\n=== A2-STYLE COMPARISON TABLE: E21 Individual (5 Models) vs Combined (1 Model) ===')
print(f'{"Category / Setup":26s} | {"Individual":>10s} | {"Combined":>10s} | {"Delta (Comb - Ind)":>18s}')
print('-' * 74)
for cat in CATEGORIES:
    print(f'{cat:26s} | {scores[cat]:10.4f} | {e21_comb[cat]:10.4f} | {(e21_comb[cat] - scores[cat]) * 100:+15.2f} pp')
print('-' * 74)
print(f'{"Category Mean (Unweighted)":26s} | {mean_ind:10.4f} | {e21_comb["mean"]:10.4f} | {(e21_comb["mean"] - mean_ind) * 100:+15.2f} pp')
print(f'{"Pooled Test Set (1,136)":26s} | {pooled_ind_raw:10.4f} | {e21_comb["pooled"]:10.4f} | {(e21_comb["pooled"] - pooled_ind_raw) * 100:+15.2f} pp')
print('=' * 74)
print(f'Saved pooled scores  -> {pooled_csv_path}')
print(f'Saved summary JSON   -> {summary_json_path}')